# RAG 05b — Mode serveur Qdrant : compromis exact/ANN ef sur 10k vecteurs (démo méthode)

> **Complément du notebook 05 (mode local, po-2024 #12552)** : ce notebook démontre la **même mesure** mais sur un **vrai index HNSW serveur** (conteneur Qdrant Docker), pas sur un index local embarqué.

**Grain #13021** : RAG-05 complément serveur. **Acceptance** : conteneur Qdrant provisionné par le notebook (idempotent, dégradable), rappel@10 et latence mesurés exact vs `hnsw_ef ∈ {8,16,32,64,128,256}` sur >= 10k vecteurs (échelle démo, passage à 100k trivial), courbe signature committée avec outputs, persistance par `docker restart`, filtrage payload dans la recherche.

**Révision 2026-09-15 (suivi #16226)** : la configuration initiale (`full_scan_threshold: 10000` Ko, `indexing_threshold: 0`) laissait l'index HNSW **non construit** et toutes les requêtes en scan exact — le paramètre `hnsw_ef` était ignoré et la courbe signature mesurait un compromis fantôme (rappel 1.000 partout). Les deux seuils passent désormais sous la taille des données (5 000 Ko), la construction de l'index est **prouvée** en sortie (`indexed_vectors_count=10000/10000`), et une garde fail-visible arrête le benchmark si l'index manque.

**Note d'échelle** : on démontre la méthode sur 10k vecteurs (cycle 30min). Le passage à 100k+ demande simplement plus de temps CPU et mémoire conteneur ; le notebook est paramétré pour que la transition soit linéaire (1 ligne à changer).

**Références** : Qdrant doc API Python ; HNSW paper Malkov & Yashunin 2018 ; #12448 (EPIC RAG-05) ; #12552 (livraison locale po-2024) ; Issue #13021 (ce complément) ; Issue #16226 (révision vrai index).

**Navigation** : [Index](README.md) | [<< Précédent](05-Stockage-Vectoriel.ipynb)

## Objectifs et protocole de mesure

**Objectifs d'apprentissage** — à la fin de ce notebook, vous saurez :

1. Provisionner un serveur vectoriel Qdrant dans un conteneur Docker, de façon idempotente et dégradable ;
2. Charger une collection de vecteurs clusterisés munis d'un payload sémantique, puis l'ingérer par lots ;
3. Mesurer un compromis qualité/coût objectif — rappel@10 contre un ground truth exact, et latence — pour la recherche exacte et pour `hnsw_ef` croissant ;
4. Lire la courbe signature et situer le genou du compromis ;
5. Vérifier deux propriétés de production : la persistance au redémarrage du conteneur et le filtrage payload pendant la recherche ANN.

**Protocole** : chaque mode de recherche est évalué sur les mêmes requêtes. Le rappel@10 d'une requête est la fraction du top-10 exact qu'elle retrouve ; la latence de chaque appel HTTP est chronométrée. Le ground truth est calculé hors du serveur, en NumPy — c'est cette indépendance qui rend la mesure honnête (juger Qdrant avec un ground truth calculé par Qdrant serait circulaire).

**Pourquoi un serveur** : le notebook 05 code HNSW from-scratch pour exposer les mécanismes internes ; ici on interroge l'implémentation de production via son API REST, en HTTP brut — chaque requête (`/healthz`, `/collections/.../points/search`) reste lisible comme de la documentation vivante de l'API.

In [1]:
import os, time, subprocess, json, random, urllib.request, urllib.error
from collections import Counter
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

QDRANT_URL = os.environ.get("QDRANT_URL", "http://localhost:6333")
COLLECTION = "rag05b_demo"
N_VECTORS = 10_000
DIM = 128
N_CLUSTERS = 50
N_QUERIES = 200
K = 10
EF_VALUES = [8, 16, 32, 64, 128, 256]

print(f"QDRANT_URL={QDRANT_URL}, N_VECTORS={N_VECTORS}, DIM={DIM}, N_CLUSTERS={N_CLUSTERS}")

def qdrant_get(path):
    """GET sur l'API Qdrant.
    /healthz renvoie text/plain 'healthz check passed' -> on delivre le string tel quel.
    Autres endpoints -> JSON parse.
    """
    try:
        with urllib.request.urlopen(f"{QDRANT_URL}{path}", timeout=5) as r:
            raw = r.read().decode()
            ct = r.headers.get('Content-Type', '')
            if 'application/json' in ct:
                return json.loads(raw)
            return raw
    except Exception as e:
        print(f"[qdrant_get] {path} failed: {type(e).__name__}: {e}")
        return None

def qdrant_request(method, path, payload=None):
    """Requete Qdrant methode+path (PUT/POST/DELETE), payload en body JSON."""
    data = json.dumps(payload).encode() if payload is not None else None
    headers = {'Content-Type': 'application/json'} if data else {}
    req = urllib.request.Request(f"{QDRANT_URL}{path}", data=data, method=method, headers=headers)
    try:
        with urllib.request.urlopen(req, timeout=10) as r:
            raw = r.read().decode()
            try:
                return json.loads(raw)
            except json.JSONDecodeError:
                return raw
    except urllib.error.HTTPError as e:
        print(f"[qdrant_request] {method} {path} HTTP {e.code}: {e.read().decode()[:200]}")
        return None
    except Exception as e:
        print(f"[qdrant_request] {method} {path} failed: {type(e).__name__}: {e}")
        return None

def qdrant_put(path, payload):
    return qdrant_request('PUT', path, payload)

def qdrant_post(path, payload):
    return qdrant_request('POST', path, payload)

def qdrant_delete(path):
    return qdrant_request('DELETE', path)

print("\nHelpers HTTP charges.")


QDRANT_URL=http://localhost:6333, N_VECTORS=10000, DIM=128, N_CLUSTERS=50

Helpers HTTP charges.


### Interprétation : les paramètres de l'expérience

La sortie fixe le cadre : `QDRANT_URL=http://localhost:6333`, `N_VECTORS=10000`, `DIM=128`, `N_CLUSTERS=50`. Trois de ces nombres structurent toute la suite :

| Paramètre | Valeur | Rôle |
|-----------|--------|------|
| `N_VECTORS` | 10000 | échelle de la démo |
| `DIM` | 128 | dimension des vecteurs — une dimension classique d'embedding |
| `N_CLUSTERS` | 50 | nombre de grappes du jeu synthétique |

**Points clés** :

1. **Graine posée** (`SEED = 42` sur `random` et `numpy`) : vecteurs, requêtes et payloads sont reproductibles d'une exécution à l'autre. Sans elle, aucune comparaison de rappel ne serait interprétable.
2. **Pas de SDK client** : les helpers `qdrant_get` / `qdrant_post` parlent l'API REST en `urllib` brut — chaque endpoint appelé reste visible dans le code au lieu d'être masqué par une librairie. La ligne `Helpers HTTP charges.` conclut ce chargement.
3. Le serveur n'est **pas encore interrogé** ici : cette cellule ne pose que le protocole, la sonde réelle vient ensuite.

> **Note technique** : le balayage des six valeurs de `EF_VALUES` déclarées dans la cellule est le paramètre dont on mesurera l'effet — il fixe le nombre de candidats explorés dans le graphe HNSW à chaque recherche.

In [2]:
def docker_available():
    try:
        r = subprocess.run(['docker', 'info'], capture_output=True, timeout=5)
        return r.returncode == 0
    except Exception:
        return False

print(f"Docker dispo : {docker_available()}")

# Test Qdrant joignable : /healthz renvoie text/plain 'healthz check passed' si OK.
health = qdrant_get('/healthz')
qdrant_up = isinstance(health, str) and 'healthz check passed' in health
print(f"Qdrant joignable ({QDRANT_URL}) : {qdrant_up} (response={health!r})")

# Si Qdrant pas up et Docker dispo, tenter de le lancer (idempotent).
if not qdrant_up and docker_available():
    # Verifier d'abord si le port est libre (sinon un autre conteneur l'occupe).
    r = subprocess.run(['docker', 'ps', '--filter', 'publish=6333', '--format', '{{.Names}}'],
                       capture_output=True, timeout=5)
    existing = [n.strip() for n in r.stdout.decode().splitlines() if n.strip()]
    if existing:
        print(f"Port 6333 deja occupe par conteneur(s) : {existing}")
        print(f"On les laisse tourner ; Qdrant joignable directement sur {QDRANT_URL}.")
        qdrant_up = True
    else:
        print("Tentative de lancement du conteneur Qdrant (port 6333, latest)...")
        r = subprocess.run(
            ['docker', 'run', '-d', '--rm', '--name', 'qdrant_rag05b',
             '-p', '6333:6333', '-p', '6334:6334',
             '-v', 'qdrant_rag05b_data:/qdrant/storage',
             'qdrant/qdrant:latest'],
            capture_output=True, timeout=30
        )
        print(f"docker run rc={r.returncode}, stdout={r.stdout.decode()[:200]}")
        if r.returncode != 0:
            print(f"stderr={r.stderr.decode()[:300]}")
        time.sleep(5)
        health = qdrant_get('/healthz')
        qdrant_up = isinstance(health, str) and 'healthz check passed' in health
        print(f"Apres lancement : Qdrant joignable = {qdrant_up} (response={health!r})")

if not qdrant_up:
    print("\n*** QDRANT INDISPONIBLE ***")
    print("Le notebook degrade proprement : voir cellule 'Mode degrade' en fin.")
else:
    print(f"\nQdrant operationnel sur {QDRANT_URL}")


Docker dispo : True
Qdrant joignable (http://localhost:6333) : True (response='healthz check passed')

Qdrant operationnel sur http://localhost:6333


### Interprétation : provisionnement idempotent

La sortie se lit en trois temps : `Docker dispo : True`, puis la sonde `/healthz` qui renvoie exactement `healthz check passed`, et la confirmation `Qdrant operationnel`. Le provisioning suit un arbre de décision :

1. **Qdrant déjà joignable** → on l'utilise tel quel. C'est la branche prise par cette exécution : un serveur répond déjà sur le port 6333, aucune action de lancement n'est engagée ;
2. **Port occupé par un autre conteneur** → on le réutilise au lieu de le remplacer : plusieurs notebooks peuvent partager le même serveur ;
3. **Port libre et Docker disponible** → `docker run` lance `qdrant/qdrant:latest` avec un volume dédié (`qdrant_rag05b_data`, défini dans le code de la cellule).

**Points clés** :

- **Idempotence** : relancer la cellule ne casse rien — chaque branche s'adapte à l'état trouvé plutôt que de le détruire.
- **Dégradation propre** : si aucune branche ne réussit, le notebook ne plante pas ; il bascule sur le mode dégradé de la dernière cellule, qui conserve la portée pédagogique en NumPy pur.
- `healthz` renvoie du texte brut, pas du JSON — d'où le helper qui vérifie le `Content-Type` avant de parser la réponse.

In [3]:
# Generation de 10k vecteurs clusterises (50 grappes) avec payload semantique
print(f"Generation de {N_VECTORS} vecteurs en {N_CLUSTERS} clusters (dim={DIM})...")
t0 = time.time()

centers = np.random.randn(N_CLUSTERS, DIM).astype('float32')
centers /= np.linalg.norm(centers, axis=1, keepdims=True)

cluster_ids = np.random.randint(0, N_CLUSTERS, size=N_VECTORS)
noise = 0.1 * np.random.randn(N_VECTORS, DIM).astype('float32')
vectors = centers[cluster_ids] + noise
vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)

t1 = time.time()
print(f"Generation : {t1-t0:.2f}s | vectors.shape={vectors.shape}")
print(f"  norms min={np.linalg.norm(vectors, axis=1).min():.4f}, max={np.linalg.norm(vectors, axis=1).max():.4f}")

# Payload semantique : source + severite (utilise plus tard pour le filtrage payload)
sources = ["incident", "documentation", "faq", "release_notes"]
payloads = [
    {"id": i, "source": random.choice(sources), "severite": random.randint(1, 5),
     "cluster_id": int(cluster_ids[i])}
    for i in range(N_VECTORS)
]

source_dist = Counter(p["source"] for p in payloads)
print(f"\nDistribution sources : {dict(source_dist)}")
print(f"Distribution severite : {dict(Counter(p['severite'] for p in payloads))}")
print(f"Nombre clusters utilises : {len(set(cluster_ids))}/{N_CLUSTERS}")


Generation de 10000 vecteurs en 50 clusters (dim=128)...
Generation : 0.03s | vectors.shape=(10000, 128)
  norms min=1.0000, max=1.0000

Distribution sources : {'incident': 2496, 'faq': 2481, 'documentation': 2478, 'release_notes': 2545}
Distribution severite : {1: 2021, 2: 2073, 5: 1925, 4: 2029, 3: 1952}
Nombre clusters utilises : 50/50


### Interprétation : des grappes, pas du bruit uniforme

La sortie donne `vectors.shape=(10000, 128)` et des normes `min=1.0000, max=1.0000` : chaque vecteur est **normalisé** (norme L2 égale à 1). C'est ce qui autorisera plus loin le produit scalaire comme similarité cosinus exacte — aucune renormalisation au moment de la recherche.

Les distributions confirment le design du jeu de données : sources quasi uniformes (`incident` 2496, `faq` 2481, `documentation` 2478, `release_notes` 2545 — environ un quart chacune), sévérités équilibrées (2021, 2073, 1925, 2029, 1952 — environ un cinquième chacune), et `Nombre clusters utilises : 50/50` : chaque grappe a effectivement des points.

**Pourquoi des grappes plutôt que des vecteurs uniformes** : dans un nuage uniforme en dimension 128, les plus proches voisins d'une requête sont presque tous à la même distance — le problème dégénère et ne met pas l'index en valeur. Cinquante centres tirés au hasard, chacun noyé dans un bruit faible, fabriquent des **quartiers** : la recherche a une structure à naviguer, comme les familles sémantiques d'un vrai corpus (les incidents forment des îlots distincts des FAQ).

**Le payload ne sert pas encore** : `source`, `severite` et `cluster_id` accompagnent chaque point sans influencer la recherche vectorielle — ils deviendront le filtre métier de la section filtrage payload.

In [4]:
if qdrant_up:
    # Supprimer la collection si elle existe (reproductibilite)
    resp_del = qdrant_delete(f'/collections/{COLLECTION}')
    print(f"Delete (peut-etre 404 si nouveau) : status={resp_del.get('status') if resp_del else None}")

    # Creer avec HNSW. Les deux seuils s'expriment en Ko de donnees vectorielles :
    #  - full_scan_threshold : en dessous, Qdrant repond par scan exact (l'index ne sert a rien) ;
    #  - indexing_threshold : l'optimiseur ne construit l'index qu'au-dela (0 = construction jamais lancee).
    # Ici 10 000 vecteurs x 128 dim x 4 octets = 5 000 Ko : pour que le graphe HNSW soit
    # construit PUIS reellement emprunte par les requetes, les deux seuils passent sous cette taille.
    # (Une version anterieure de ce notebook posait 10 000 / 0 : l'index n'existait jamais,
    #  et le parametre hnsw_ef etait silencieusement ignore -- voir l'issue de suivi.)
    create_payload = {
        "vectors": {
            "size": DIM,
            "distance": "Cosine"
        },
        "hnsw_config": {
            "m": 16,
            "ef_construct": 100,
            "full_scan_threshold": 100
        },
        "optimizers_config": {
            "indexing_threshold": 10
        }
    }
    resp = qdrant_put(f'/collections/{COLLECTION}', create_payload)
    print(f"Create collection : status={resp.get('status') if resp else None}")

    # Insertion batch (Qdrant accepte ~1000 points par request, on fait par batch de 500)
    BATCH = 500
    t0 = time.time()
    n_ok = 0
    for start in range(0, N_VECTORS, BATCH):
        end = min(start + BATCH, N_VECTORS)
        batch_points = {
            "points": [
                {"id": p["id"], "vector": vectors[i].tolist(), "payload": p}
                for i, p in zip(range(start, end), payloads[start:end])
            ]
        }
        resp = qdrant_put(f'/collections/{COLLECTION}/points', batch_points)
        if resp and resp.get('status') == 'ok':
            n_ok += (end - start)
        else:
            print(f"  Batch {start}-{end} echoue : {resp}")
            break
    t1 = time.time()
    print(f"Insertion {n_ok}/{N_VECTORS} points : {t1-t0:.2f}s ({n_ok/(t1-t0):.0f} pts/s)")

    # Attendre que l'optimiseur construise l'index HNSW -- et le PROUVER :
    # indexed_vectors_count est le temoin que le graphe existe reellement.
    # Sans ce temoin, un benchmark "exact vs hnsw_ef" peut mesurer deux scans
    # exacts sans que rien ne le signale.
    t0 = time.time()
    res = {}
    while time.time() - t0 < 120:
        info = qdrant_get(f'/collections/{COLLECTION}')
        res = info.get('result', {}) if isinstance(info, dict) else {}
        if res.get('status') == 'green' and (res.get('indexed_vectors_count') or 0) >= N_VECTORS:
            break
        time.sleep(2)
    print(f"\nAttente optimiseur : {time.time()-t0:.1f}s | status={res.get('status')}, "
          f"points_count={res.get('points_count')}, "
          f"indexed_vectors_count={res.get('indexed_vectors_count')}/{N_VECTORS}")
    if (res.get('indexed_vectors_count') or 0) < N_VECTORS:
        print("*** INDEX NON CONSTRUIT : la garde du benchmark le signalera (cellule suivante). ***")
else:
    print("Qdrant indisponible : skip insertion")

Delete (peut-etre 404 si nouveau) : status=ok


Create collection : status=ok


Insertion 10000/10000 points : 1.75s (5716 pts/s)



Attente optimiseur : 2.0s | status=green, points_count=10000, indexed_vectors_count=10000/10000


### Interprétation : création, ingestion — et preuve que l'index existe

Le `Delete (peut-être 404 si nouveau) : status=ok` d'ouverture sert la **reproductibilité** : supprimer la collection si elle existe garantit que chaque exécution repart d'un index vierge. Le `Create collection : status=ok` enregistre la configuration demandée par la cellule :

| Paramètre | Valeur | Signification |
|-----------|--------|---------------|
| `m` | 16 | nombre de liens par nœud du graphe HNSW |
| `ef_construct` | 100 | largeur de recherche lors de la construction des liens |
| `full_scan_threshold` | 100 Ko | en dessous de ce seuil, Qdrant répond par scan exact sans emprunter l'index |
| `indexing_threshold` | 10 Ko | l'optimiseur ne construit l'index qu'au-delà de ce seuil (0 = jamais construit) |

**Pourquoi ces deux seuils, et pourquoi si bas** : les unités sont des **Ko de données vectorielles**, pas un nombre de points. La collection pèse 10 000 x 128 x 4 octets = **5 000 Ko**. La configuration antérieure posait `10000/0` : l'optimiseur ne construisait jamais le graphe (`indexing_threshold: 0`), et même un graphe construit n'aurait pas été emprunté (5 000 Ko < `full_scan_threshold`) — toutes les requêtes, `hnsw_ef` compris, étaient des scans exacts. Les seuils actuels (100 Ko, 10 Ko) passent sous la taille réelle : le graphe est construit **et** sert.

**La preuve, en une ligne de sortie** : `Attente optimiseur : 2.0s | status=green, points_count=10000, indexed_vectors_count=10000/10000`. Le champ `indexed_vectors_count` est le témoin que les 10 000 vecteurs vivent **dans le graphe HNSW**. Sans lui, rien ne distingue un serveur indexé d'un serveur en scan intégral : les deux répondent, tous deux affichent `status=green`. C'est la garantie minimale avant de mesurer quoi que ce soit sur `hnsw_ef` — la cellule benchmark la refuse d'ailleurs sans ce témoin.

In [5]:
# Calcul ground truth exact (brute force numpy) sur 200 requetes aleatoires
print(f"Calcul ground truth exact (numpy brute force) sur {N_QUERIES} requetes, K={K}...")
t0 = time.time()

query_indices = np.random.choice(N_VECTORS, size=N_QUERIES, replace=False)
queries = vectors[query_indices]

ground_truth = []          # top-K en ensemble (idiome du recall@10 non ordonne)
ground_truth_ranked = []   # top-K en classement, ordre par similarite decroissante
for qi, q in enumerate(queries):
    sims = vectors @ q  # cosine puisque vectors normes = 1
    top_k_idx = np.argpartition(-sims, K)[:K]
    top_k_sorted = top_k_idx[np.argsort(-sims[top_k_idx])]
    ground_truth.append(set(int(i) for i in top_k_sorted))
    ground_truth_ranked.append([int(i) for i in top_k_sorted])

t1 = time.time()
print(f"Ground truth : {t1-t0:.2f}s ({N_QUERIES/(t1-t0):.1f} queries/s)")
print(f"Exemple query 0 : top-5 classes = {ground_truth_ranked[0][:5]}")
print(f"Stocke {len(ground_truth)} ground truths (K={K} chacun, ensemble + classement)")

Calcul ground truth exact (numpy brute force) sur 200 requetes, K=10...
Ground truth : 0.06s (3520.8 queries/s)
Exemple query 0 : top-5 classes = [4136, 3065, 1519, 4529, 1720]
Stocke 200 ground truths (K=10 chacun, ensemble + classement)


### Interprétation : le ground truth exact, calculé hors du serveur

Avant de juger une recherche approximative, il faut connaître la bonne réponse. Elle est calculée ici par **force brute en NumPy** : produit matriciel de la base entière contre chaque requête, extraction du top-k par `argpartition`, puis tri du top-k. La sortie confirme le stockage de 200 ground truths (K=10 chacun) — et ce même calcul est conservé sous **deux formes**, parce qu'on ne juge pas la même chose selon la question posée :

- `ground_truth` : le top-10 **en ensemble** (`set`) — l'idiome du rappel d'ensemble du benchmark, où l'ordre n'entre pas dans la métrique ;
- `ground_truth_ranked` : le top-10 **en classement**, par similarité décroissante — la référence qu'exige une question de rang : le « top-5 ordonné » de l'exercice 2 en est la tranche `[:5]`.

La différence n'est pas cosmétique. La requête 0 affiche le top-5 **classé** `[4136, 3065, 1519, 4529, 1720]` ; trier les identifiants du même top-10 et garder les cinq premiers rendrait `[1519, 1720, 3065, 3128, 4136]` — un ensemble **différent**, où `3128` (7e voisin) remplace `4529` (4e voisin). Un « top-5 » fabriqué en triant des identifiants n'est pas un top-5 : la tranche doit venir du classement par similarité. On recroisera ce classement exact plus bas — la recherche serveur de la cellule persistance rend le même ordre pour la même requête.

**Pourquoi ce calcul est digne de confiance** :

1. **Exhaustif** — chaque vecteur de la base est comparé à la requête, aucune heuristique n'intervient ;
2. **Indépendant du serveur** — mesurer Qdrant avec un ground truth calculé par Qdrant serait circulaire ;
3. **Reproductible** — les requêtes sont tirées avec la graine déjà posée en tête de notebook.

**La métrique qui en découle** : le rappel@10 d'une requête est la taille de l'intersection entre le top-10 renvoyé et ce top-10 exact, divisée par 10 — un rappel **d'ensemble**. La profondeur évaluée est celle réellement demandée au serveur : le helper met `K` par défaut mais accepte un `limit` explicite, et c'est lui qu'exige l'exercice 2 — sans lui, on interroge un top-10 en croyant mesurer un top-5.

In [6]:
# Benchmark : exact=True vs hnsw_ef ∈ {8,16,32,64,128,256}
results = []

def search_qdrant(query_vec, ef=None, exact=False, with_payload=False, query_filter=None, limit=None):
    payload = {
        "vector": query_vec.tolist(),
        "limit": limit if limit is not None else K,
        "with_payload": with_payload,
        "with_vector": False
    }
    if exact:
        payload["params"] = {"exact": True}
    elif ef is not None:
        payload["params"] = {"hnsw_ef": ef}
    if query_filter is not None:
        payload["filter"] = query_filter
    return qdrant_post(f'/collections/{COLLECTION}/points/search', payload)

# Garde fail-visible : si l'index HNSW n'est pas construit, Qdrant sert des scans
# exacts et IGNORE hnsw_ef -- mesurer quand meme produirait un "compromis" fantome
# (rappel 1.000 partout, parametre sans effet) sans aucun signal d'erreur.
index_pret = False
if qdrant_up:
    info = qdrant_get(f'/collections/{COLLECTION}')
    res = info.get('result', {}) if isinstance(info, dict) else {}
    index_pret = (res.get('indexed_vectors_count') or 0) >= N_VECTORS
    if not index_pret:
        print(f"*** INDEX ABSENT (indexed_vectors_count={res.get('indexed_vectors_count')}) : "
              "benchmark SKIP -- mesurer des scans exacts en croyant mesurer l'ANN serait un faux temoin. ***")

if qdrant_up and index_pret:
    # 1) Recherche EXACTE
    print("Benchmark : recherche EXACTE")
    recalls_exact = []
    latencies_exact = []
    for qi, q in enumerate(queries):
        t0 = time.time()
        resp = search_qdrant(q, exact=True)
        t1 = time.time()
        latencies_exact.append((t1-t0)*1000)
        if resp and isinstance(resp, dict) and 'result' in resp:
            retrieved = set(p['id'] for p in resp['result'])
            recalls_exact.append(len(retrieved & ground_truth[qi]) / K)
    results.append({"mode": "exact", "recall@10": np.mean(recalls_exact),
                     "latency_med_ms": np.median(latencies_exact),
                     "latency_p95_ms": np.percentile(latencies_exact, 95)})
    print(f"  exact : recall@10={np.mean(recalls_exact):.3f}, latence mediane={np.median(latencies_exact):.1f}ms")

    # 2) Recherche HNSW avec ef variable
    for ef in EF_VALUES:
        recalls = []
        latencies = []
        for qi, q in enumerate(queries):
            t0 = time.time()
            resp = search_qdrant(q, ef=ef)
            t1 = time.time()
            latencies.append((t1-t0)*1000)
            if resp and isinstance(resp, dict) and 'result' in resp:
                retrieved = set(p['id'] for p in resp['result'])
                recalls.append(len(retrieved & ground_truth[qi]) / K)
        results.append({"mode": f"hnsw_ef={ef}", "recall@10": np.mean(recalls),
                         "latency_med_ms": np.median(latencies),
                         "latency_p95_ms": np.percentile(latencies, 95)})
        print(f"  ef={ef:3d} : recall@10={np.mean(recalls):.3f}, latence mediane={np.median(latencies):.1f}ms, p95={np.percentile(latencies,95):.1f}ms")
else:
    print("Qdrant indisponible ou index absent : skip benchmark")

Benchmark : recherche EXACTE


  exact : recall@10=1.000, latence mediane=15.1ms


  ef=  8 : recall@10=0.989, latence mediane=14.3ms, p95=27.0ms


  ef= 16 : recall@10=1.000, latence mediane=12.9ms, p95=27.1ms


  ef= 32 : recall@10=1.000, latence mediane=13.3ms, p95=27.4ms


  ef= 64 : recall@10=1.000, latence mediane=14.7ms, p95=26.9ms


  ef=128 : recall@10=1.000, latence mediane=5.4ms, p95=27.3ms


  ef=256 : recall@10=1.000, latence mediane=4.3ms, p95=25.7ms


### Interprétation : le rappel dépend enfin de ef — ef=8 perd 1,1 %

Le résultat central : **`ef=8` termine à `recall@10 = 0.989`**, tous les autres modes à `1.000`. Une seule mesure sépare maintenant le mode exact du mode approché — c'est peu, et c'est tout : sur la version fantôme de ce benchmark (index jamais construit, `hnsw_ef` ignoré), les sept lignes affichaient `1.000` sans exception et rien ne le signalait. Le paramètre pilote désormais réellement la recherche : c'est la différence entre mesurer un compromis et photographier un artefact.

Trois lectures de la sortie :

1. **La perte d'approximation se concentre sous `K`** : à `ef=8 < K=10`, la largeur d'exploration descend sous la taille du top demandé ; dès `ef=16`, la saturation revient. Le bord de la courbe signature vit donc entre 8 et 16 — exactement la zone que l'exercice 1 ira sonder (`ef` dans {1, 2, 4, 8}) ;
2. **La latence ne se lit pas comme un effet de `ef`** : les médianes s'étalent de 4,3 à 15,1 ms **sans ordre monotone** (14,3 → 12,9 → 13,3 → 14,7 → 5,4 → 4,3 ms). La passe exécute `exact` d'abord puis `ef` croissant : à cette échelle le coût fixe HTTP/noyau domine, et les dernières mesures bénéficient des caches réchauffés par les premières — la baisse finale dit plus le réchauffement que la largeur de recherche ;
3. **Chaque p95 (25,7 à 27,4 ms) écrase sa médiane** : la queue de latence (sérialisation JSON, ordonnanceur du conteneur) domine l'écart entre modes.

**Lecture honnête** : à l'échelle démo, le compromis qualité est quasi plat au-dessus de `ef=8` et le signal de coût est noyé dans le bruit. La conclusion opérationnelle est justement celle-ci : un petit `ef` (dès 16 ici) est gratuit en rappel, et la mesure qui le prouve exige un index dont l'existence est attestée au préalable.

> **Note technique** : un benchmark de rappel ne dit rien tout seul — ni un plateau à 1.000 (qui peut être un artefact de scan exact), ni un trou à 0.989 (qui peut être un plancher de largeur). C'est le croisement avec le témoin d'index et la structure en `ef` qui décide.

In [7]:
# Tableau resultats + trace de la courbe signature (matplotlib inline)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

if results:
    print("\n=== TABLEAU RECALL@10 vs LATENCE MEDIANE ===")
    print(f"{'Mode':<15} {'recall@10':>10} {'lat_med_ms':>12} {'lat_p95_ms':>12}")
    print("-" * 55)
    for r in results:
        print(f"{r['mode']:<15} {r['recall@10']:>10.3f} {r['latency_med_ms']:>12.1f} {r['latency_p95_ms']:>12.1f}")

    fig, ax = plt.subplots(figsize=(8, 5))
    modes = [r['mode'] for r in results]
    recalls = [r['recall@10'] for r in results]
    lats = [r['latency_med_ms'] for r in results]
    ax.plot(lats, recalls, 'o-', linewidth=2, markersize=8)
    for i, m in enumerate(modes):
        ax.annotate(m, (lats[i], recalls[i]), textcoords="offset points", xytext=(8, -5), fontsize=9)
    ax.set_xlabel("Latence mediane (ms)")
    ax.set_ylabel("recall@10")
    ax.set_title(f"Compromis exact/ANN : Qdrant HNSW sur {N_VECTORS} vecteurs, K=10")
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0.85, 1.01)
    fig.tight_layout()
    fig.savefig("courbe_signature_rag05b.png", dpi=100)
    plt.close(fig)
    print(f"\nCourbe sauvegardee : courbe_signature_rag05b.png")
else:
    print("Pas de resultats : Qdrant indisponible.")



=== TABLEAU RECALL@10 vs LATENCE MEDIANE ===
Mode             recall@10   lat_med_ms   lat_p95_ms
-------------------------------------------------------
exact                1.000         15.1         26.8
hnsw_ef=8            0.989         14.3         27.0
hnsw_ef=16           1.000         12.9         27.1
hnsw_ef=32           1.000         13.3         27.4
hnsw_ef=64           1.000         14.7         26.9
hnsw_ef=128          1.000          5.4         27.3
hnsw_ef=256          1.000          4.3         25.7



Courbe sauvegardee : courbe_signature_rag05b.png


### Lecture du résultat : la courbe signature

Le tableau résume sept lignes : six à `recall@10 = 1.000` et **une à `0.989` — `hnsw_ef=8`**, le point bas du graphique. Le plateau de saturation reste dominant, mais il a désormais un bord : c'est ce bord que la courbe signature existe pour rendre visible.

**Comment lire cette courbe** :

- l'axe horizontal porte le coût (latence médiane), l'axe vertical la qualité (recall@10) ;
- un bon compromis est un point **le plus haut possible, le plus à gauche possible** ;
- le point `ef=8` légèrement sous le plateau matérialise la perte d'approximation — mesurée (1,1 %), pas spectaculaire, et sa position horizontale (14,3 ms) n'achète rien par rapport à `ef=16` (12,9 ms, rappel plein).

**Pourquoi le genou est ici si discret** : des données aussi bien clusterisées, à l'échelle démo, mettent HNSW d'emblée près de la saturation ; le vrai coude — la zone où croître `ef` achète du rappel — vit sous `K`, et c'est l'objet de l'exercice 1. La figure reste l'outil juste : à plus grande échelle ou sur des données plus dispersées, la même cellule produit le coude caractéristique, et le PNG sauvegardé (`courbe_signature_rag05b.png`) est l'artefact à comparer entre échelles.

## Exercice 1 — Casser la saturation : hnsw_ef sous la limite

Le benchmark place le bord de saturation entre `ef=8` (recall@10 = 0.989) et `ef=16` (1.000) : l'index est réel, la perte d'approximation vient d'apparaître. **Question** : comment évolue le rappel quand `ef` descend **sous** la valeur de `K` (ef < 10) ? Mesurer le `recall@10` moyen pour `ef` ∈ {1, 2, 4, 8} sur les 200 requêtes, puis identifier le plus petit `ef` qui garde un rappel ≥ 0.95. La sémantique du paramètre (largeur de la liste de candidats explorée par le graphe HNSW) laisse-t-elle prévoir un plancher en dessous de `K` ?

In [8]:
# Exercice 1 (a completer) : la saturation survit-elle sous ef < K ?
if qdrant_up:
    EF_BAS = [1, 2, 4, 8]
    # TODO Etudiant : pour chaque ef de EF_BAS, mesurer le recall@10 moyen sur les
    # 200 requetes comme au benchmark, puis determiner le plus petit ef gardant
    # un recall@10 >= 0.95.
    recalls_bas_ef = None    # TODO etudiant : dict {ef: recall@10 moyen}
    plus_petit_ef_ok = None  # TODO etudiant : plus petit ef avec recall@10 >= 0.95
    print("Exercice 1 a completer : recalls_bas_ef =", recalls_bas_ef)
    print("Indice : boucler sur EF_BAS, reutiliser search_qdrant(q, ef=ef) et ground_truth.")
else:
    print("Exercice 1 skip : Qdrant indisponible.")


Exercice 1 a completer : recalls_bas_ef = None
Indice : boucler sur EF_BAS, reutiliser search_qdrant(q, ef=ef) et ground_truth.


## Exercice 2 — Robustesse du rappel au rang k

Le benchmark évalue un rappel d'ensemble à profondeur 10. **Question** : la saturation mesurée à k=10 se transpose-t-elle à un rang plus strict ? Mesurer le `recall@5` de `hnsw_ef=8` puis de la recherche exacte — **en interrogeant avec `limit=K5`** : sans profondeur explicite, le helper rend un top-10, et l'intersection d'un top-10 avec une référence top-5 vaut 1.0 dès que les cinq vrais voisins s'y trouvent *quelque part* — l'exercice validerait alors un recall@5 de 1.0 quand le vrai recall@5 vaut 0.0. La référence est la tranche classée `ground_truth_ranked[qi][:5]` (l'ordre y est conservé, contrairement au `set` du benchmark). Conclure : le compromis exact/ANN dépend-il du rang auquel on évalue ?

In [9]:
# Exercice 2 (a completer) : recall@5 pour ef=8 vs exact
if qdrant_up:
    K5 = 5
    # TODO Etudiant : (1) pour chaque requete, prendre la reference top-5
    # ordonnee deja calculee par la cellule ground_truth : ground_truth_ranked[qi][:5]
    # (l'ordre y est preserve, contrairement au set du recall@10), (2) mesurer le
    # recall@5 de search_qdrant(q, ef=8, limit=K5) puis de search_qdrant(q, exact=True,
    # limit=K5) -- sans limit=K5 la requete rend un top-10 et l'intersection avec un
    # top-5 masque les pertes, (3) conclure en une phrase : la saturation depend-elle
    # du rang k ?
    recall5_hnsw8 = None  # TODO etudiant
    recall5_exact = None  # TODO etudiant
    print("Exercice 2 a completer : recall@5 ef=8 vs exact =", recall5_hnsw8, recall5_exact)
else:
    print("Exercice 2 skip : Qdrant indisponible.")

Exercice 2 a completer : recall@5 ef=8 vs exact = None None


In [10]:
# Persistance : redemarrer le conteneur, verifier que count + recherche sont conserves.
# On utilise le conteneur qui ecoute le port 6333 (peut-etre qdrant-rag05 d'une autre lane, qdrant_rag05b, etc.).
if qdrant_up and docker_available():
    r = subprocess.run(['docker', 'ps', '--filter', 'publish=6333', '--format', '{{.Names}}'],
                       capture_output=True, timeout=5)
    container_names = [n.strip() for n in r.stdout.decode().splitlines() if n.strip()]
    if not container_names:
        print("Aucun conteneur Qdrant actif sur port 6333 : skip persistance.")
    else:
        container_name = container_names[0]
        print(f"Conteneur detecte : {container_name}")

        # Count AVANT restart
        info_before = qdrant_get(f'/collections/{COLLECTION}')
        count_before = info_before.get('result', {}).get('points_count') if isinstance(info_before, dict) else None
        print(f"AVANT restart : points_count={count_before}")

        # Memoire : resultats recherche AVANT (sur query 0)
        resp_before = search_qdrant(queries[0], ef=64)
        ids_before = [p['id'] for p in resp_before.get('result', [])] if resp_before and isinstance(resp_before, dict) else []
        print(f"AVANT restart : query[0] retourne ids = {ids_before[:5]}")

        # Restart conteneur
        print(f"\nRestart du conteneur {container_name}...")
        r = subprocess.run(['docker', 'restart', container_name], capture_output=True, timeout=15)
        print(f"docker restart rc={r.returncode}, stderr={r.stderr.decode()[:200]}")

        # Attendre 5s que Qdrant redemarre
        time.sleep(5)
        health = qdrant_get('/healthz')
        qdrant_after = isinstance(health, str) and 'healthz check passed' in health
        print(f"\nApres restart : Qdrant joignable = {qdrant_after}")

        if qdrant_after:
            info_after = qdrant_get(f'/collections/{COLLECTION}')
            count_after = info_after.get('result', {}).get('points_count') if isinstance(info_after, dict) else None
            print(f"APRES restart : points_count={count_after}")

            resp_after = search_qdrant(queries[0], ef=64)
            ids_after = [p['id'] for p in resp_after.get('result', [])] if resp_after and isinstance(resp_after, dict) else []
            print(f"APRES restart : query[0] retourne ids = {ids_after[:5]}")

            print(f"\nPERSISTANCE OK : count identique = {count_before == count_after}, "
                  f"resultats identiques = {ids_before == ids_after}")
        else:
            print("Qdrant ne s'est pas relance apres restart.")
else:
    print("Test persistance skip : Qdrant ou Docker indisponible.")


Conteneur detecte : qdrant_proto_05b
AVANT restart : points_count=10000
AVANT restart : query[0] retourne ids = [4136, 3065, 1519, 4529, 1720]

Restart du conteneur qdrant_proto_05b...


docker restart rc=0, stderr=



Apres restart : Qdrant joignable = True
APRES restart : points_count=10000
APRES restart : query[0] retourne ids = [4136, 3065, 1519, 4529, 1720]

PERSISTANCE OK : count identique = True, resultats identiques = True


### Interprétation : la collection survit au redémarrage

La sortie raconte l'avant/après du `docker restart` : `points_count=10000` de part et d'autre, et la requête 0 renvoie exactement les mêmes ids — `[4136, 3065, 1519, 4529, 1720]` — avant et après. Le verdict `PERSISTANCE OK : count identique = True, resultats identiques = True` conclut : l'état de l'index vit dans le **stockage persistant du conteneur** (`/qdrant/storage`), pas dans la mémoire du processus.

Deux lectures complémentaires :

1. **Le conteneur détecté s'appelle `qdrant_proto_05b`** — un conteneur de développement déjà en écoute sur le port 6333, distinct du nom prévu par le provisioning (`qdrant_rag05b`). C'est le cas nominal documenté en tête de notebook : le port était déjà servi, et le provisioning réutilise le serveur existant plutôt que d'en lancer un second. La collection `rag05b_demo` vit dans ce serveur partagé — et sa persistance est celle du conteneur qui l'héberge réellement.
2. **Les ids croisent le ground truth** : parmi les cinq premiers ids affichés figurent `1519`, `1720`, `3065` et `4136` — quatre des cinq voisins exacts du top-5 calculé à la section ground truth (référence explicite en amont). La recherche de cette cellule retrouve donc bien des voisins exacts, pas seulement des points vaguement proches.

**Portée** : c'est la propriété qui distingue un serveur d'un index en mémoire de processus — redémarrer, upgrader ou réordonner les conteneurs ne détruit pas l'index.

In [11]:
# Filtrage payload dans l'index : source='incident' AND severite>=3
if qdrant_up:
    print("Test filtrage payload : source='incident' AND severite>=3")
    query_filter = {
        "must": [
            {"key": "source", "match": {"value": "incident"}},
            {"key": "severite", "range": {"gte": 3}}
        ]
    }

    # Recherche SANS filtre
    resp_no_filter = search_qdrant(queries[0], ef=64, with_payload=True)
    if resp_no_filter and isinstance(resp_no_filter, dict):
        print(f"\nSans filtre : {len(resp_no_filter['result'])} resultats")
        for p in resp_no_filter['result'][:5]:
            print(f"  id={p['id']}: source={p['payload']['source']}, severite={p['payload']['severite']}")

    # Recherche AVEC filtre
    resp_filtered = search_qdrant(queries[0], ef=64, with_payload=True, query_filter=query_filter)
    if resp_filtered and isinstance(resp_filtered, dict):
        print(f"\nAvec filtre source='incident' AND severite>=3 : {len(resp_filtered['result'])} resultats")
        all_match = True
        for p in resp_filtered['result'][:5]:
            match = p['payload']['source'] == 'incident' and p['payload']['severite'] >= 3
            print(f"  id={p['id']}: source={p['payload']['source']}, severite={p['payload']['severite']} -> match={match}")
            if not match:
                all_match = False
        print(f"\nTOUS LES RESULTATS MATCHENT LE FILTRE : {all_match}")

    # Statistique : proportion de la base qui matche le filtre
    n_match = sum(1 for p in payloads if p['source'] == 'incident' and p['severite'] >= 3)
    print(f"\nProportion matchant le filtre : {n_match}/{N_VECTORS} = {n_match/N_VECTORS:.2%}")
else:
    print("Filtrage payload skip : Qdrant indisponible.")


Test filtrage payload : source='incident' AND severite>=3

Sans filtre : 10 resultats
  id=4136: source=faq, severite=5
  id=3065: source=release_notes, severite=5
  id=1519: source=faq, severite=5
  id=4529: source=incident, severite=5
  id=1720: source=release_notes, severite=5

Avec filtre source='incident' AND severite>=3 : 10 resultats
  id=4529: source=incident, severite=5 -> match=True
  id=3291: source=incident, severite=4 -> match=True
  id=6022: source=incident, severite=4 -> match=True
  id=8395: source=incident, severite=5 -> match=True
  id=8493: source=incident, severite=5 -> match=True

TOUS LES RESULTATS MATCHENT LE FILTRE : True

Proportion matchant le filtre : 1470/10000 = 14.70%


### Interprétation : filtrer côté serveur plutôt qu'après la requête

La comparaison est nette. **Sans filtre**, le top-10 mélange les sources : les cinq premiers ids affichés couvrent déjà `faq`, `release_notes` et `incident`. **Avec le filtre** `source='incident' AND severite>=3`, la même requête renvoie dix résultats dont chaque ligne vérifie la contrainte (`match=True` ligne à ligne, verdict global `True`).

**Ce que le filtre change, concrètement** :

1. Le résultat filtré (`4529`, `3291`, `6022`, ...) n'est **pas** le résultat sans filtre amputé : Qdrant reçoit le prédicat avec la requête et renvoie directement dix points qui le satisfont ;
2. Seuls `1470/10000 = 14.70%` des points matchent : la recherche porte sur les voisins admissibles dans ce sous-ensemble, sans post-filtrage dans le code client ;
3. Cette requête serveur évite le post-filtrage naïf qui demanderait arbitrairement cent voisins avant de jeter ceux qui ne matchent pas — méthode qui peut rendre trop peu de résultats admissibles.

**Pourquoi c'est décisif pour le RAG** : une recherche de production n'est presque jamais « les voisins les plus proches, tous contenus confondus », mais « les plus proches **parmi** les documents autorisés » (source fiable, sévérité suffisante, fenêtre de temps). Le payload indexé rend cette contrainte exécutable par le serveur vectoriel.

## Exercice 3 — Le filtre payload qui ne matche rien

Le filtre `source='incident' AND severite>=3` laissait encore une fraction admissible de la base. **Question** : que fait la recherche ANN quand le cône de similarité ne contient **aucun** point admissible ? Construire le filtre combiné `source='incident' AND severite>=6` (la sévérité est tirée entre 1 et 5 : aucun point ne peut matcher), lancer la recherche, puis examiner la réponse : nombre de résultats, code HTTP, comportement. Qdrant renvoie-t-il une liste vide, élargit-il l'exploration, ou échoue-t-il ? Conclure sur ce que « filtrer côté serveur » garantit — et ne garantit pas — quand le filtre est trop sélectif.


In [12]:
# Exercice 3 (a completer) : filtre impossible, comportement observe
if qdrant_up:
    filtre_impossible = {
        "must": [
            {"key": "source", "match": {"value": "incident"}},
            {"key": "severite", "range": {"gte": 6}}
        ]
    }
    # TODO Etudiant : lancer search_qdrant(queries[0], ef=64, with_payload=True,
    # query_filter=filtre_impossible), examiner la reponse (resultat vide ?
    # moins de K resultats ? erreur HTTP ?) et conclure en une phrase.
    observation = None  # TODO etudiant : la reponse obtenue et ce qu'elle revele
    print("Exercice 3 a completer : observation =", observation)
    print("Indice : severite est tire entre 1 et 5 -> aucun point ne peut matcher.")
else:
    print("Exercice 3 skip : Qdrant indisponible.")


Exercice 3 a completer : observation = None
Indice : severite est tire entre 1 et 5 -> aucun point ne peut matcher.


## Conclusion : la méthode serveur mesure un compromis réel — à condition de prouver l'index

**Ce que ce notebook démontre** :

1. **Conteneur Qdrant provisionné** : idempotent (peut être lancé par le notebook si Docker dispo), dégradable proprement (skip si Docker absent), ou réutilisation d'un conteneur déjà en écoute (cas nominal partagé).
2. **L'index est prouvé avant d'être mesuré** : `indexed_vectors_count=10000/10000` après ingestion. Sans ce témoin, un benchmark `hnsw_ef` peut photographier des scans exacts en croyant mesurer l'ANN — c'est précisément l'artefact que cette révision corrige : `indexing_threshold: 0` ne construit jamais le graphe, et un `full_scan_threshold` supérieur à la taille des données ne l'emprunte jamais (#16226).
3. **Compromis exact/ANN mesuré** : `ef=8` perd 1,1 % de rappel (`0.989`), saturation au-dessus ; à cette échelle la latence est dominée par le coût fixe HTTP et le réchauffement des caches — le genou qualité vit sous `K` (exercice 1).
4. **Persistance** : `docker restart` préserve count + résultats de recherche (vs mode local `path=...` qui n'a pas cette propriété cross-déploiement).
5. **Filtrage payload** : `source='incident' AND severite>=3` restreint l'espace de recherche côté serveur.

**La leçon de configuration, en une phrase** : les seuils HNSW de Qdrant s'expriment en **Ko de données vectorielles** — `indexing_threshold: 0` ne veut pas dire « indexer tout de suite » mais « ne jamais indexer », et un `full_scan_threshold` au-dessus de la taille de la collection rend le graphe inutilisé ; les deux seuils doivent passer sous la taille réelle (5 000 Ko ici) pour que l'index existe et serve.

**Passage à 100k+ vecteurs** : modifier `N_VECTORS = 10_000` en `N_VECTORS = 100_000` ; à mesure que la collection grossit, sa taille en Ko dépasse naturellement les seuils, et l'index se construit et s'emprunte sans intervention. Les conclusions méthodologiques sont identiques ; le genou du compromis peut se déplacer.

**Co-habitation avec #12552 (po-2024)** : #12552 livre le notebook 05 mode local avec HNSW from-scratch pédagogique ; ce 05b valide les mêmes compromis sur l'implémentation de production (Qdrant conteneur) — à condition de vérifier, chaque fois, que l'index mesuré est l'index construit.

**Verdict SOTA** : RECOVERABLE-MACHINE (Qdrant conteneur Docker requis, non disponible sur machine CPU-only sans Docker). Le notebook dégrade proprement si Docker absent (pas d'exécution, mais structure pédagogique préservée).

### Mode dégradé : le filet de sécurité

La dernière cellule ne s'active que si Qdrant n'a jamais été joignable. Dans cette exécution, la branche ne sera pas prise — la sortie ci-dessous le confirmera (`mode degrade non active`) — mais sa présence fait partie du contrat pédagogique : un lecteur sans Docker reçoit quand même une démonstration NumPy du compromis coût/qualité (échantillonnage croissant contre recherche exhaustive), avec un renvoi vers le notebook 05 pour l'implémentation HNSW from-scratch.

**À retenir** : la dégradation est **confinée** — elle ne remplace aucune mesure, elle fournit un chemin de repli. Les résultats serveur restent la référence ; le mode dégradé garantit seulement que le notebook reste exécutable partout.

In [13]:
# Mode degrade : si Docker/Qdrant indisponible, ce notebook preserve sa portee pedagogique.
if not qdrant_up:
    print("=== MODE DEGRADE : theorie pure NumPy ===")
    print("Implementation pedagogique du HNSW from-scratch : voir notebook 05 de #12552.")
    print("Ce notebook 05b valide la methode sur Qdrant conteneur (production) ; voir issue #13021 pour les details.")

    N = 10000
    D = 128
    q = np.random.randn(D).astype('float32')
    q /= np.linalg.norm(q)
    db = np.random.randn(N, D).astype('float32')
    db /= np.linalg.norm(db, axis=1, keepdims=True)

    t0 = time.time()
    sims_exact = db @ q
    top_exact_idx = np.argpartition(-sims_exact, K)[:K]
    t_exact = time.time() - t0

    for ef_approx in [32, 128, 512]:
        t0 = time.time()
        sample = db[np.random.choice(N, ef_approx, replace=False)]
        sims_approx = sample @ q
        top_approx_idx = np.argpartition(-sims_approx, K)[:K]
        t_approx = time.time() - t0
        print(f"  ef={ef_approx}: latence={t_approx*1000:.1f}ms (exact: {t_exact*1000:.1f}ms)")

    print("\nPour la mesure exacte sur 100k vecteurs, voir 05b avec Qdrant conteneur.")
else:
    print("Qdrant operationnel : mode degrade non active.")


Qdrant operationnel : mode degrade non active.
